In [15]:
!pip install -r requirements.txt

In [16]:
import os
import sys
from dotenv import load_dotenv
from llama_stack_client import LlamaStackClient
import pandas as pd
import logging
import requests
from io import BytesIO

In [17]:
sys.path.append('..')
# Load environment variables from .env file
load_dotenv()

logger = logging.getLogger(__name__)
logger.setLevel("INFO")

# Initialize the Llama Stack client
client = LlamaStackClient(
    base_url=os.getenv("LLAMA_STACK_SERVER_URL", "http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321")
)

#file_path = "data/Commercial-Direct-LATAM-USD-Q3-2025-Subscriptions.csv"
file_path = "Commercial-Direct-LATAM-USD-Q3-2025-Subscriptions.md"
url = "https://www.openshift.guide/openshift-guide-screen.pdf"
vector_db_skus_name = "skus_rh_vector_db"
vector_db_ocp_name = "ocp_rh_vector_db"

logger.info("Connected to Llama Stack server")

INFO:__main__:Connected to Llama Stack server


In [18]:
for m in client.models.list():
    print(f"Model: {m}")
    #if m.id == "sentence-transformers/nomic-ai/nomic-embed-text-v1.5":
        #client.models.unregister(
        #    model_id=m.id
        #)

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/models "HTTP/1.1 200 OK"


Model: Model(id='sentence-transformers/ibm-granite/granite-embedding-125m-english', created=1779405778, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'sentence-transformers', 'provider_resource_id': 'ibm-granite/granite-embedding-125m-english', 'embedding_dimension': 768}, object='model')
Model: Model(id='vllm-inference/redhataillama-31-8b-instruct', created=1779405778, owned_by='llama_stack', custom_metadata={'model_type': 'llm', 'provider_id': 'vllm-inference', 'provider_resource_id': 'redhataillama-31-8b-instruct'}, object='model')
Model: Model(id='sentence-transformers/nomic-ai/nomic-embed-text-v1.5', created=1779405778, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'sentence-transformers', 'provider_resource_id': 'nomic-ai/nomic-embed-text-v1.5', 'embedding_dimension': 768}, object='model')


In [19]:
vector_stores = client.vector_stores.list()

print(f"Vector stores {vector_stores}")

for vector_store in vector_stores:
    client.vector_stores.delete(
        vector_store_id=vector_store.id
    )
    print(f"Vector store: {vector_store.name} deleted")

print("All Vector stores deleted")

vector_stores = client.vector_stores.list()

print(f"Vector stores {vector_stores}")

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: DELETE http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores/vs_4e800ce5-334c-4377-8d3f-bbbe6e7f39e8 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Vector stores SyncOpenAICursorPage[VectorStore](data=[VectorStore(id='vs_4e800ce5-334c-4377-8d3f-bbbe6e7f39e8', created_at=1779403878, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0), expires_after=None, expires_at=None, last_active_at=1779403878, metadata={'provider_id': 'milvus-remote', 'provider_vector_store_id': 'vs_4e800ce5-334c-4377-8d3f-bbbe6e7f39e8', 'embedding_model': 'sentence-transformers/ibm-granite/granite-embedding-125m-english', 'embedding_dimension': '768'}, name='skus_rh_vector_db', object='vector_store', status='completed', usage_bytes=0)], has_more=False, last_id='vs_4e800ce5-334c-4377-8d3f-bbbe6e7f39e8', object='list', first_id='vs_4e800ce5-334c-4377-8d3f-bbbe6e7f39e8')
Vector store: skus_rh_vector_db deleted
All Vector stores deleted
Vector stores SyncOpenAICursorPage[VectorStore](data=[], has_more=False, last_id=None, object='list', first_id=None)


In [20]:
# Create a vector store skus and index the file
vector_store_skus = client.vector_stores.create(
    name=vector_db_skus_name,
    extra_body={
        "provider_id": "milvus-remote",
        "embedding_model": "sentence-transformers/ibm-granite/granite-embedding-125m-english",
        "embedding_dimension": 768,
    },
)

# Create a vector store ocp and index the file
vector_store_ocp = client.vector_stores.create(
    name=vector_db_ocp_name,
    extra_body={
        "provider_id": "milvus-remote",
        "embedding_model": "sentence-transformers/ibm-granite/granite-embedding-125m-english",
        "embedding_dimension": 768,
    },
)

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


In [21]:
vector_db_skus_id = ""
vector_dbs = client.vector_stores.list()
for vector_db in vector_dbs:
    if vector_db.name == vector_db_skus_name:
        vector_db_skus_id = vector_db.id
        break
if vector_db_skus_id == "":
    print(f"Vector DB ID for SKUs: {vector_db_skus_name} not found in the vector stores")

print(f"Vector DB ID for SKUs: {vector_db_skus_id}")

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Vector DB ID for SKUs: vs_16ed7c1b-6ece-429b-8523-6c529c5122bc


In [22]:
# Upload a document
file = client.files.create(
    file=open(file_path, "rb"),
    purpose="assistants",
)
print(f"Uploaded: {file.id}")

client.vector_stores.files.create(
    vector_store_id=vector_db_skus_id,
    file_id=file.id,
    attributes={
        "document_id": "Subscriptions.md",
        "source": file_path
    },
    chunking_strategy={
        "type": "static",
        "static": {"max_chunk_size_tokens": 512, "chunk_overlap_tokens": 128},
    },
)

print(f"File {file.id} loaded into Vector store SKUs with ID: {vector_db_skus_id}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/files "HTTP/1.1 200 OK"


Uploaded: file-79c19783e3f04b44a42b5a56454d1d33


INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores/vs_16ed7c1b-6ece-429b-8523-6c529c5122bc/files "HTTP/1.1 200 OK"


File file-79c19783e3f04b44a42b5a56454d1d33 loaded into Vector store SKUs with ID: vs_16ed7c1b-6ece-429b-8523-6c529c5122bc


In [23]:
vector_db_ocp_id = ""
vector_dbs = client.vector_stores.list()
for vector_db in vector_dbs:
    if vector_db.name == vector_db_ocp_name:
        vector_db_ocp_id = vector_db.id
        break
if vector_db_skus_id == "":
    print(f"Vector DB ID for SKUs: {vector_db_ocp_name} not found in the vector stores")

print(f"Vector DB ID for SKUs: {vector_db_ocp_id}")

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Vector DB ID for SKUs: vs_c83f94a1-129c-4432-baa5-f04ac5618059


In [24]:
response = requests.get(url)
file_buffer = BytesIO(response.content)
file_buffer.name = "openshift-guide-screen.pdf"

# Upload a document
file_url = client.files.create(
    file=file_buffer,
    purpose="assistants",
)
print(f"Uploaded: {file_url.id}")

client.vector_stores.files.create(
    vector_store_id=vector_db_ocp_id,
    file_id=file_url.id,
    attributes={
        "document_id": file_buffer.name,
        "source": url
    },
    chunking_strategy={
        "type": "static",
        "static": {"max_chunk_size_tokens": 512, "chunk_overlap_tokens": 128},
    },
)

print(f"File {file_url.id} loaded into Vector store OCP with ID: {vector_db_ocp_id}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/files "HTTP/1.1 200 OK"


Uploaded: file-2f19e0d83f974aa18c2923e3f093f749


INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores/vs_c83f94a1-129c-4432-baa5-f04ac5618059/files "HTTP/1.1 200 OK"


File file-2f19e0d83f974aa18c2923e3f093f749 loaded into Vector store OCP with ID: vs_c83f94a1-129c-4432-baa5-f04ac5618059


In [25]:
vector_stores = client.vector_stores.list()

print(f"Vector stores created {vector_stores}")

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Vector stores created SyncOpenAICursorPage[VectorStore](data=[VectorStore(id='vs_16ed7c1b-6ece-429b-8523-6c529c5122bc', created_at=1779405784, file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1), expires_after=None, expires_at=None, last_active_at=1779405784, metadata={'provider_id': 'milvus-remote', 'provider_vector_store_id': 'vs_16ed7c1b-6ece-429b-8523-6c529c5122bc', 'embedding_model': 'sentence-transformers/ibm-granite/granite-embedding-125m-english', 'embedding_dimension': '768'}, name='skus_rh_vector_db', object='vector_store', status='completed', usage_bytes=0), VectorStore(id='vs_c83f94a1-129c-4432-baa5-f04ac5618059', created_at=1779405784, file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1), expires_after=None, expires_at=None, last_active_at=1779405784, metadata={'provider_id': 'milvus-remote', 'provider_vector_store_id': 'vs_c83f94a1-129c-4432-baa5-f04ac5618059', 'embedding_model': 'sentence-transformers/ibm-grani

In [28]:
query = "Can you search and list the SKU, SKU_Description, List_Price and Currency of Product Advanced Cluster Management"

# Ask questions with file search
response = client.responses.create(
    model="vllm-inference/redhataillama-31-8b-instruct",
    input=query,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_db_skus_id],
    }],
)

logger.info(f"RAG Query from {vector_db_skus_name} - Result: \n{response.output_text}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/responses "HTTP/1.1 200 OK"
INFO:__main__:RAG Query from skus_rh_vector_db - Result: 
Based on the search results, the SKU, SKU_Description, List_Price and Currency of Product Advanced Cluster Management are:

* SKU: MCT3945, MCT3946, MCT3968, MCT3969
* SKU_Description: Red Hat Advanced Cluster Management for Kubernetes, Premium (2 Core or 4 vCPU), Red Hat Advanced Cluster Management for Kubernetes, Standard (2 Core or 4 vCPU), Red Hat Advanced Cluster Management for Kubernetes for Distributed Computing (Edge Server), Premium (2 Core or 4 vCPU), Red Hat Advanced Cluster Management for Kubernetes for Distributed Computing (Edge Server), Standard (2 Core or 4 vCPU)
* List_Price: 1.100,00 USD, 743,00 USD, 330,00 USD, 220,00 USD
* Currency: USD

Please note that the prices and SKUs may vary depending on the region and country. Cite sources immediately at the end of sentences usin

In [ ]:
query = "What is Red Hat OpenShift?"

# Ask questions with file search
response = client.responses.create(
    model="vllm-inference/redhataillama-31-8b-instruct",
    input=query,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_db_ocp_id],
    }],
)

logger.info(f"RAG Query from {vector_db_ocp_name} - Result: \n{response.output_text}")